# YallaMotor Used Cars Web Scraping
## Haya Hany Abo Said
## 220230264

## 1. Import Libraries

In [18]:
import glob
import pandas as pd
from bs4 import BeautifulSoup

## 2. Scraping Function

In [21]:
def ScrapData(soup):
    PageData = []
    car_titles = soup.find_all("a", class_=lambda x: x and "line-clamp-2" in x)
    print("Total Cars Found:", len(car_titles))

    for title_tag in car_titles:
        card = title_tag.find_parent("div", class_=lambda x: x and "cursor-pointer" in x)
        if card is None:
            continue

        # Title
        title = title_tag.get_text(" ", strip=True)

        # Brand and Model
        clean_title = title.replace("مستعملة", "").replace("...", "").strip()
        words = clean_title.split()
        brand = words[0] if len(words) > 0 else None
        model_words = []
        
        for word in words[1:]:
            if word.isdigit() and 1990 <= int(word) <= 2026:
                break
        
            model_words.append(word)
        
        model = " ".join(model_words).strip()

        # Price
        price_tag = card.find("div", class_=lambda x: x and "text-main" in x and "text-lg" in x)

        price = (price_tag.get_text(" ", strip=True) if price_tag else None)

        # Other details
        mileage = None
        location = None
        year = None

        for span in card.find_all("span"):
            text = span.get_text(" ", strip=True)

            # KM
            if "كم" in text and any(char.isdigit() for char in text):
                mileage = text

            # Year
            elif text.isdigit():
                number = int(text)
                if 1990 <= number <= 2026:
                    year = text

            # Location
            elif text in [
                "دبي",
                "أبو ظبي",
                "الشارقة",
                "عجمان",
                "العين",
                "رأس الخيمة",
                "الفجيرة",
                "أم القيوين"
            ]:
                location = text

        # One car record
        result = {
            "Title": title,
            "Brand": brand,
            "Model": model,
            "Year": year,
            "KM": mileage,
            "Location": location,
            "Price": price
        }

        PageData.append(result)

    return PageData

## 3. Test Scraping on One Page

In [22]:
with open("toyota_1.html", "r", encoding="utf-8") as f:
    soup_test = BeautifulSoup(f.read(), "html.parser")

test_data = ScrapData(soup_test)

test_data[:5]

Total Cars Found: 12


[{'Title': 'تويوتا لاند كروز برادو 20...',
  'Brand': 'تويوتا',
  'Model': 'لاند كروز برادو 20',
  'Year': '2020',
  'KM': '67,225 كم',
  'Location': 'دبي',
  'Price': '122,799 درهم'},
 {'Title': 'تويوتا Corolla Cross 2025...',
  'Brand': 'تويوتا',
  'Model': 'Corolla Cross',
  'Year': '2025',
  'KM': '70,543 كم',
  'Location': 'دبي',
  'Price': '84,999 درهم'},
 {'Title': 'تويوتا يارس 2022 مستعملة...',
  'Brand': 'تويوتا',
  'Model': 'يارس',
  'Year': '2022',
  'KM': '72,930 كم',
  'Location': 'دبي',
  'Price': '34,299 درهم'},
 {'Title': 'تويوتا لاند كروز برادو 20...',
  'Brand': 'تويوتا',
  'Model': 'لاند كروز برادو 20',
  'Year': '2022',
  'KM': '87,000 كم',
  'Location': 'دبي',
  'Price': '157,000 درهم'},
 {'Title': 'تويوتا كامري 2024 مستعملة...',
  'Brand': 'تويوتا',
  'Model': 'كامري',
  'Year': '2024',
  'KM': '72,000 كم',
  'Location': 'دبي',
  'Price': '109,999 درهم'}]

## 4. Collect Data from All Saved Pages

In [23]:
html_files = glob.glob("*.html")

all_data = []

for file in html_files:
    with open(file, "r", encoding="utf-8") as f:
        soup_page = BeautifulSoup(f.read(), "html.parser")

    page_data = ScrapData(soup_page)

    all_data.extend(page_data)

    print(file, "->", len(page_data), "cars")

print("\nTotal before duplicates:", len(all_data))

Total Cars Found: 12
bmw_1.html -> 12 cars
Total Cars Found: 12
bmw_2.html -> 12 cars
Total Cars Found: 12
bmw_3.html -> 12 cars
Total Cars Found: 12
ford_1.html -> 12 cars
Total Cars Found: 12
ford_2.html -> 12 cars
Total Cars Found: 12
ford_3.html -> 12 cars
Total Cars Found: 12
hyundai_1.html -> 12 cars
Total Cars Found: 12
hyundai_2.html -> 12 cars
Total Cars Found: 12
hyundai_3.html -> 12 cars
Total Cars Found: 12
mercedes-benz_1.html -> 12 cars
Total Cars Found: 12
mercedes-benz_2.html -> 12 cars
Total Cars Found: 12
mercedes-benz_3.html -> 12 cars
Total Cars Found: 12
nissan_1.html -> 12 cars
Total Cars Found: 12
nissan_2.html -> 12 cars
Total Cars Found: 12
nissan_3.html -> 12 cars
Total Cars Found: 12
toyota_1.html -> 12 cars
Total Cars Found: 12
toyota_2.html -> 12 cars
Total Cars Found: 12
toyota_3.html -> 12 cars

Total before duplicates: 216


## 5. Create DataFrame

In [25]:
df = pd.DataFrame(all_data)

df.reset_index(drop=True, inplace=True)

print("Total cars:", len(df))
print("Shape:", df.shape)

df.head(10)

Total cars: 216
Shape: (216, 7)


,Title,Brand,Model,Year,KM,Location,Price
0,بي إم دبليو X2 2022 مستعم...,بي,إم دبليو X2,2022,"64,027 كم",دبي,"61,699 درهم"
1,بي إم دبليو اكس1 2024 مست...,بي,إم دبليو اكس1,2024,"42,432 كم",دبي,"134,999 درهم"
2,بي إم دبليو اكس6 2023 مست...,بي,إم دبليو اكس6,2023,"64,298 كم",دبي,"194,999 درهم"
3,بي إم دبليو M8 Competitio...,بي,إم دبليو M8 Competitio,2023,"11,396 كم",دبي,"339,999 درهم"
4,بي إم دبليو اكس1 2018 مست...,بي,إم دبليو اكس1,2018,"117,873 كم",دبي,"45,299 درهم"
5,بي إم دبليو X2 2020 مستعم...,بي,إم دبليو X2,2020,"66,545 كم",دبي,"58,099 درهم"
6,بي إم دبليو X7 2020 مستعم...,بي,إم دبليو X7,2020,"40,109 كم",دبي,"172,999 درهم"
7,بي إم دبليو 2 سيريز كوبيه...,بي,إم دبليو 2 سيريز كوبيه,2024,"11,000 كم",دبي,"98,999 درهم"
8,بي إم دبليو 2 سيريز كوبيه...,بي,إم دبليو 2 سيريز كوبيه,2023,"91,000 كم",دبي,"119,999 درهم"
9,بي إم دبليو اكس1 2024 مست...,بي,إم دبليو اكس1,2024,"6,400 كم",دبي,"139,999 درهم"


## 6. Save Raw Dataset

In [26]:
df.to_csv(
    "yallamotor_used_cars_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Raw dataset saved.")

Raw dataset saved.
